In [ ]:
%pip install git+https://github.com/redwoodresearch/Easy-Transformer.git
%pip install einops datasets transformers fancy_einsum

  Cloning https://github.com/redwoodresearch/Easy-Transformer.git to /tmp/pip-req-build-gzyc4if3
  Running command git clone --filter=blob:none --quiet https://github.com/redwoodresearch/Easy-Transformer.git /tmp/pip-req-build-gzyc4if3
  Resolved https://github.com/redwoodresearch/Easy-Transformer.git to commit ea15315dd24481e9e2ac5c3ef335d82907a1dc34
  Preparing metadata (setup.py) ... done


# Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 Small
 <h1><b>Intro</b></h1>
 This notebook implements all experiments in our paper (which is available on arXiv).
 For background on the task, see the paper.
 Refer to the demo of the <a href="https://github.com/neelnanda-io/Easy-Transformer">Easy-Transformer</a> library here: <a href="https://github.com/neelnanda-io/Easy-Transformer/blob/main/EasyTransformer_Demo.ipynb">demo with ablation and patching</a>.

 Reminder of the circuit:
 <img src="https://i.imgur.com/arokEMj.png">

# Setup

In [ ]:
from copy import deepcopy
import torch
assert torch.cuda.device_count() == 1
from tqdm import tqdm
import pandas as pd
import torch
import torch as t
from easy_transformer.EasyTransformer import (
    EasyTransformer,
)
from time import ctime
from functools import partial

import numpy as np
from tqdm import tqdm
import pandas as pd

from easy_transformer.experiments import (
    ExperimentMetric,
    AblationConfig,
    EasyAblation,
    EasyPatching,
    PatchingConfig,
)
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import random
import einops
from IPython import get_ipython
from copy import deepcopy
from easy_transformer.ioi_dataset import (
    IOIDataset,
)
from easy_transformer.ioi_utils import (
    path_patching,
    max_2d,
    CLASS_COLORS,
    show_pp,
    show_attention_patterns,
    scatter_attention_and_contribution,
)
from random import randint as ri
from easy_transformer.ioi_circuit_extraction import (
    do_circuit_extraction,
    get_heads_circuit,
    CIRCUIT,
)
from easy_transformer.ioi_utils import logit_diff, probs
from easy_transformer.ioi_utils import get_top_tokens_and_probs as g

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    try:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")
    except ModuleNotFoundError:
        print("Autoreload not available on this Python version, continuing without it.")


Autoreload not available on this Python version, continuing without it.


 Initialise model (use larger N or fewer templates for no warnings about in-template ablation)

In [ ]:
model = EasyTransformer.from_pretrained("gpt2").cuda()
model.set_use_attn_result(True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/easy_transformer/components.py:616: UserWarning: Moved LN1 to the attention block
  warnings.warn("Moved LN1 to the attention block")


Moving model to device:  cuda
Finished loading pretrained model gpt2 into EasyTransformer!


 Initialise dataset

In [ ]:
N = 250
ioi_dataset = IOIDataset(
    prompt_type="BABA",
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
)  # TODO make this a seeded dataset

print(f"Here are two of the prompts from the dataset: {ioi_dataset.sentences[:2]}")

Here are two of the prompts from the dataset: ['After the lunch, Matthew and Andrew went to the hospital. Matthew gave a basketball to Andrew', 'Then, Alicia and Jason were thinking about going to the garden. Alicia wanted to give a drink to Jason']


 See logit difference

In [ ]:
model.reset_hooks()
model_logit_diff = logit_diff(model, ioi_dataset, all=True)
model_io_probs = probs(model, ioi_dataset)
print(
    # f"The model gets average logit difference {model_logit_diff.item()} over {N} examples"
 )
print(f"The model gets average IO probs {model_io_probs.item()} over {N} examples")


The model gets average IO probs 0.48391836881637573 over 250 examples


 The circuit

In [ ]:
circuit = deepcopy(CIRCUIT)

# we make the ABC dataset in order to knockout other model components
abc_dataset = (  # TODO seeded
    ioi_dataset.gen_flipped_prompts(("IO", "RAND"))
    .gen_flipped_prompts(("S", "RAND"))
    .gen_flipped_prompts(("S1", "RAND"))
)
# we then add hooks to the model to knockout all the heads except the circuit
model.reset_hooks()
model, _ = do_circuit_extraction(
    model=model,
    heads_to_keep=get_heads_circuit(ioi_dataset=ioi_dataset, circuit=circuit),
    mlps_to_remove={},
    ioi_dataset=ioi_dataset,
    mean_dataset=abc_dataset,
)
circuit_logit_diff = logit_diff(model, ioi_dataset, all=True)  # if this returns a vector
circuit_logit_diff_mean = circuit_logit_diff.mean()
print(
    f"The circuit gets average logit difference {circuit_logit_diff_mean.item()} over {N} examples"
)
model.reset_hooks()

/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_dataset.py:501: UserWarning: S2 index has been computed as the same for S and S2
  warnings.warn("S2 index has been computed as the same for S and S2")


The circuit gets average logit difference 3.7632670402526855 over 250 examples


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# Assuming model_logit_diff and circuit_logit_diff are already defined and processed as in your example

fig = px.scatter(
    x=model_logit_diff.detach().cpu().numpy(),
    y=circuit_logit_diff.detach().cpu().numpy(),
)

# Generate x values from -1 to 9
x_values = np.linspace(-1, 9, 100)

# Add y=x line
fig.add_trace(go.Scatter(x=x_values, y=x_values, mode='lines', name='y=x'))

# Show the figure
fig.show() # ABBA


# Path patching

In [ ]:
def plot_path_patching_small(
    model,
    ioi_dataset,
    receiver_hooks,
    position,
):
    from tqdm import tqdm
    model.reset_hooks()
    default_logit_diff = logit_diff(model, ioi_dataset)
    results = torch.zeros(size=(12, 12))

    # Only a subset of layers and heads to avoid OOM
    layers_to_test = [0, 5, 11]
    heads_to_test = [0, 5, 11]

    for source_layer in tqdm(layers_to_test):
        for source_head_idx in heads_to_test:
            model.reset_hooks()
            model = path_patching(
                model=model,
                D_new=abc_dataset[:10],      # only first 10 examples
                D_orig=ioi_dataset[:10],     # only first 10 examples
                sender_heads=[(source_layer, source_head_idx)],
                receiver_hooks=receiver_hooks,
                positions=[position],
                return_hooks=False,
                freeze_mlps=False,
                have_internal_interactions=False,
            )
            cur_logit_diff = logit_diff(model, ioi_dataset[:10])
            results[source_layer][source_head_idx] = (
                cur_logit_diff - default_logit_diff
            )

    results /= default_logit_diff
    results *= 100
    fig = show_pp(
        results,
        title=f"Effect of patching (Heads->Final Residual Stream State) path (small)",
        return_fig=True,
        show_fig=False,
        bartitle="% change in logit difference",
    )
    fig.show()


## compute faithfullnes, minimality  & completeness


In [ ]:
from copy import deepcopy
import pandas as pd
from IPython.display import display

# ============================================================
# 0. Mean-ablation dataset (as in IOI code)
# ============================================================
abc_dataset = (
    ioi_dataset.gen_flipped_prompts(("IO", "RAND"))
    .gen_flipped_prompts(("S", "RAND"))
    .gen_flipped_prompts(("S1", "RAND"))
)
mean_dataset = abc_dataset

# ============================================================
# 1. Build canonical "full circuit" dict for heads
#    full_heads_dict: {(layer, head): positions_list}
# ============================================================
circuit = deepcopy(CIRCUIT)
full_heads_dict = get_heads_circuit(ioi_dataset=ioi_dataset, circuit=circuit)
all_head_keys = list(full_heads_dict.keys())  # list of (layer, head)

def F_logit_diff(model):
    ld = logit_diff(model, ioi_dataset, all=True)
    return ld.mean().item()

def F_keep(head_keys_subset):
    """
    Circuit-only style: keep only heads in head_keys_subset (subset of all_head_keys),
    mean-ablating all other attention heads.
    """
    model.reset_hooks()
    heads_to_keep = {k: full_heads_dict[k] for k in head_keys_subset}
    model_cfg, _ = do_circuit_extraction(
        model=model,
        heads_to_keep=heads_to_keep,
        mlps_to_remove={},
        ioi_dataset=ioi_dataset,
        mean_dataset=mean_dataset,
    )
    F_val = F_logit_diff(model_cfg)
    model.reset_hooks()
    return F_val

def F_remove(head_keys_subset):
    """
    Full-model-minus-K: remove (mean-ablated) exactly heads in head_keys_subset,
    leaving all other heads untouched.
    """
    model.reset_hooks()
    heads_to_remove = {k: full_heads_dict[k] for k in head_keys_subset}
    model_cfg, _ = do_circuit_extraction(
        model=model,
        heads_to_remove=heads_to_remove,
        mlps_to_remove={},
        ioi_dataset=ioi_dataset,
        mean_dataset=mean_dataset,
    )
    F_val = F_logit_diff(model_cfg)
    model.reset_hooks()
    return F_val

# ============================================================
# 2. Faithfulness: |F(M) - F(C)|
# ============================================================
model.reset_hooks()
F_M = F_logit_diff(model)                # full model M
print(f"[Faithfulness] F(M) full model: {F_M:.4f}")

F_C = F_keep(all_head_keys)              # circuit C = all circuit heads only
print(f"[Faithfulness] F(C) circuit-only: {F_C:.4f}")

faithfulness_gap = abs(F_M - F_C)
fraction_kept = F_C / F_M
print(f"[Faithfulness] |F(M) - F(C)| = {faithfulness_gap:.4f}")
print(f"[Faithfulness] F(C)/F(M) = {fraction_kept:.3%}")

# ============================================================
# 3. Completeness (paper metric): |F(C\K) - F(M\K)|
#    Here we use K = whole circuit class G (2nd bullet in text).
# ============================================================

comp_rows = []
for G, heads_G in circuit.items():
    if G == "ablation":
        continue
    heads_G = list(heads_G)                         # list of (layer, head)
    K = set(heads_G)                                # K ⊆ C

    # C\K: keep all circuit heads except those in K
    C_minus_K_keys = [h for h in all_head_keys if h not in K]
    F_C_minus_K = F_keep(C_minus_K_keys)

    # M\K: full model with K removed
    F_M_minus_K = F_remove(heads_G)

    incompleteness = abs(F_C_minus_K - F_M_minus_K)
    comp_rows.append({
        "removed_group": G,
        "F_C_minus_K": F_C_minus_K,
        "F_M_minus_K": F_M_minus_K,
        "incompleteness": incompleteness,
    })

df_compl = pd.DataFrame(comp_rows)

print("\n[Completeness] class-based K = G, incompleteness = |F(C\\K) - F(M\\K)|:")
display(df_compl.sort_values("incompleteness", ascending=False))

# ============================================================
# 4. Minimality (paper-flavored): for each v in class G,
#    choose K = C\G  (so C\K = G),
#    minimality(v) = |F(C\(K∪{v})) - F(C\K)|
#                  = |F(G\\{v}) - F(G)|.
# ============================================================

min_rows = []

for G, heads_G in circuit.items():
    if G == "ablation":
        continue
    heads_G = list(heads_G)

    # K = C\G  => C\K = G.  So F(C\K) = F(G) = circuit restricted to heads_G.
    F_G = F_keep(heads_G)

    for v in heads_G:
        heads_G_minus_v = [h for h in heads_G if h != v]
        F_G_minus_v = F_keep(heads_G_minus_v)
        minimality_score = abs(F_G_minus_v - F_G)

        min_rows.append({
            "class": G,
            "head": str(v),
            "F_G": F_G,
            "F_G_minus_v": F_G_minus_v,
            "minimality": minimality_score,
        })

df_min = pd.DataFrame(min_rows)

print("\n[Minimality] per-head scores minimality(v) = |F(G\\{v}) - F(G)|:")
display(df_min.sort_values("minimality", ascending=False))

print("\n[Minimality] per-class summary (mean/max as % of F(M)):")
df_min_summary = df_min.groupby("class")["minimality"].agg(["mean", "max"]).reset_index()
df_min_summary["mean_%_of_FM"] = 100 * df_min_summary["mean"] / F_M
df_min_summary["max_%_of_FM"]  = 100 * df_min_summary["max"]  / F_M
display(df_min_summary.sort_values("max", ascending=False))


[Faithfulness] F(M) full model: 3.6267
[Faithfulness] F(C) circuit-only: 3.7202
[Faithfulness] |F(M) - F(C)| = 0.0935
[Faithfulness] F(C)/F(M) = 102.579%

[Completeness] class-based K = G, incompleteness = |F(C\K) - F(M\K)|:


,removed_group,F_C_minus_K,F_M_minus_K,incompleteness
5,previous token,1.509916,3.052624,1.542708
4,duplicate token,0.700482,1.713248,1.012766
1,negative,5.774208,6.508296,0.734088
2,s2 inhibition,0.097605,-0.470186,0.567791
0,name mover,0.908585,1.292796,0.384212
3,induction,1.335144,1.414637,0.079493



[Minimality] per-head scores minimality(v) = |F(G\{v}) - F(G)|:


,class,head,F_G,F_G_minus_v,minimality
0,name mover,"(9, 9)",-3.089893,-2.327171,0.762722
12,negative,"(11, 10)",0.784039,0.422804,0.361234
8,name mover,"(9, 7)",-3.089893,-3.346464,0.256571
4,name mover,"(10, 6)",-3.089893,-2.863129,0.226764
11,negative,"(10, 7)",0.784039,0.570244,0.213795
6,name mover,"(10, 1)",-3.089893,-2.891844,0.198049
16,s2 inhibition,"(8, 10)",0.433275,0.237771,0.195504
2,name mover,"(9, 6)",-3.089893,-2.923638,0.166255
14,s2 inhibition,"(7, 9)",0.433275,0.269394,0.163881
3,name mover,"(10, 10)",-3.089893,-2.928122,0.161771



[Minimality] per-class summary (mean/max as % of F(M)):


,class,mean,max,mean_%_of_FM,max_%_of_FM
2,name mover,0.206299,0.762722,5.688385,21.030950
3,negative,0.287515,0.361234,7.927803,9.960514
5,s2 inhibition,0.102225,0.195504,2.818718,5.390732
0,duplicate token,0.000000,0.000000,0.000000,0.000000
1,induction,0.000000,0.000000,0.000000,0.000000
4,previous token,0.000000,0.000000,0.000000,0.000000


In [ ]:
plot_path_patching_small(
    model,
    ioi_dataset,
    receiver_hooks=[(f"blocks.{model.cfg.n_layers-1}.hook_resid_post", None)],
    position="end",
)


  0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_dataset.py:769: UserWarning:

Some groups have less than 5 prompts, they have lengths [1, 2, 1, 1, 1, 1, 1, 1, 1]

/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_dataset.py:501: UserWarning:

S2 index has been computed as the same for S and S2

100%|██████████| 3/3 [00:02<00:00,  1.45it/s]


# Writing direction results
 (change the layer_no and head_no)

In [ ]:
scatter_attention_and_contribution(
    model=model, layer_no=9, head_no=9, ioi_dataset=ioi_dataset
)

# Copy score
 For NMs and Negative NMs

In [ ]:
def check_copy_circuit(model, layer, head, ioi_dataset, verbose=False, neg=False):
    cache = {}
    model.cache_some(cache, lambda x: x == "blocks.0.hook_resid_post")
    model(ioi_dataset.toks.long())
    if neg:
        sign = -1
    else:
        sign = 1
    z_0 = model.blocks[1].attn.ln1(cache["blocks.0.hook_resid_post"])

    v = torch.einsum("eab,bc->eac", z_0, model.blocks[layer].attn.W_V[head])
    v += model.blocks[layer].attn.b_V[head].unsqueeze(0).unsqueeze(0)

    o = sign * torch.einsum("sph,hd->spd", v, model.blocks[layer].attn.W_O[head])
    logits = model.unembed(model.ln_final(o))

    k = 5
    n_right = 0

    for seq_idx, prompt in enumerate(ioi_dataset.ioi_prompts):
        for word in ["IO", "S", "S2"]:
            pred_tokens = [
                model.tokenizer.decode(token)
                for token in torch.topk(
                    logits[seq_idx, ioi_dataset.word_idx[word][seq_idx]], k
                ).indices
            ]
            if "S" in word:
                name = "S"
            else:
                name = word
            if " " + prompt[name] in pred_tokens:
                n_right += 1
            else:
                if verbose:
                    print("-------")
                    print("Seq: " + ioi_dataset.sentences[seq_idx])
                    print("Target: " + ioi_dataset.ioi_prompts[seq_idx][name])
                    print(
                        " ".join(
                            [
                                f"({i+1}):{model.tokenizer.decode(token)}"
                                for i, token in enumerate(
                                    torch.topk(
                                        logits[
                                            seq_idx, ioi_dataset.word_idx[word][seq_idx]
                                        ],
                                        k,
                                    ).indices
                                )
                            ]
                        )
                    )
    percent_right = (n_right / (ioi_dataset.N * 3)) * 100
    print(
        f"Copy circuit for head {layer}.{head} (sign={sign}) : Top {k} accuracy: {percent_right}%"
    )
    return percent_right


neg_sign = False
print(" --- Name Mover heads --- ")
check_copy_circuit(model, 9, 9, ioi_dataset, neg=neg_sign)
check_copy_circuit(model, 10, 0, ioi_dataset, neg=neg_sign)
check_copy_circuit(model, 9, 6, ioi_dataset, neg=neg_sign)

neg_sign = True
print(" --- Negative heads --- ")
check_copy_circuit(model, 10, 7, ioi_dataset, neg=neg_sign)
check_copy_circuit(model, 11, 10, ioi_dataset, neg=neg_sign)

neg_sign = False
print(" ---  Random heads for control ---  ")
check_copy_circuit(
    model, random.randint(0, 11), random.randint(0, 11), ioi_dataset, neg=neg_sign
)
check_copy_circuit(
    model, random.randint(0, 11), random.randint(0, 11), ioi_dataset, neg=neg_sign
)
check_copy_circuit(
    model, random.randint(0, 11), random.randint(0, 11), ioi_dataset, neg=neg_sign
)

 --- Name Mover heads --- 
Copy circuit for head 9.9 (sign=1) : Top 5 accuracy: 100.0%
Copy circuit for head 10.0 (sign=1) : Top 5 accuracy: 96.66666666666667%
Copy circuit for head 9.6 (sign=1) : Top 5 accuracy: 96.26666666666667%
 --- Negative heads --- 
Copy circuit for head 10.7 (sign=-1) : Top 5 accuracy: 100.0%
Copy circuit for head 11.10 (sign=-1) : Top 5 accuracy: 100.0%
 ---  Random heads for control ---  
Copy circuit for head 5.0 (sign=1) : Top 5 accuracy: 0.26666666666666666%
Copy circuit for head 2.2 (sign=1) : Top 5 accuracy: 0.0%
Copy circuit for head 9.10 (sign=1) : Top 5 accuracy: 0.0%


0.0

# S-Inhibition patching

In [ ]:
def plot_path_patching_small(
    model,
    ioi_dataset,
    receiver_hooks,   # list[(hook_name, head_idx or None)]
    position: str,
):
    from tqdm import tqdm
    model.reset_hooks()
    default_logit_diff = logit_diff(model, ioi_dataset[:10])  # use subset to save RAM

    results = torch.zeros(size=(12, 12))

    # choose which sender heads to scan
    layers_to_test = [0, 5, 11]          # or range(12) if your GPU survives
    heads_to_test  = [0, 5, 11]

    for source_layer in tqdm(layers_to_test):
        for source_head_idx in heads_to_test:
            model.reset_hooks()
            path_patching(
                model=model,
                D_new=abc_dataset[:10],     # corrupted data (small subset)
                D_orig=ioi_dataset[:10],    # clean data (small subset)
                sender_heads=[(source_layer, source_head_idx)],
                receiver_hooks=receiver_hooks,
                positions=[position],
                return_hooks=False,
                freeze_mlps=False,
                have_internal_interactions=False,
            )

            cur_logit_diff = logit_diff(model, ioi_dataset[:10])
            results[source_layer, source_head_idx] = (
                cur_logit_diff - default_logit_diff
            )

    results /= default_logit_diff
    results *= 100

    fig = show_pp(
        results,
        title=f"Effect of patching (Heads->{position}) path (small)",
        return_fig=True,
        show_fig=False,
        bartitle="% change in logit difference",
    )
    fig.show()



In [ ]:
plot_path_patching_small(
    model,
    ioi_dataset,
    receiver_hooks=[
        (f"blocks.{layer_idx}.attn.hook_v", head_idx)
        for layer_idx, head_idx in circuit["s2 inhibition"]
    ],
    position="S2",
)

/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_dataset.py:769: UserWarning:

Some groups have less than 5 prompts, they have lengths [1, 2, 1, 1, 1, 1, 1, 1, 1]

  0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_dataset.py:501: UserWarning:

S2 index has been computed as the same for S and S2

 67%|██████▋   | 2/3 [00:01<00:00,  1.58it/s]/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_utils.py:1117: UserWarning:

Torch all close for blocks.7.attn.hook_v

/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_utils.py:1117: UserWarning:

Torch all close for blocks.8.attn.hook_v

100%|██████████| 3/3 [00:01<00:00,  1.58it/s]


# Attention probs of NMs

In [ ]:
ys = []
average_attention = {}

for idx, dataset in enumerate([ioi_dataset, abc_dataset]):
    fig = go.Figure()
    for heads_raw in circuit["name mover"][:3]:
        heads = [heads_raw]
        average_attention[heads_raw] = {}
        cur_ys = []
        cur_stds = []
        att = torch.zeros(size=(dataset.N, dataset.max_len, dataset.max_len))
        for head in tqdm(heads):
            att += show_attention_patterns(
                model, [head], dataset, return_mtx=True, mode="attn"
            )
        att /= len(heads)

        vals = att[torch.arange(dataset.N), ioi_dataset.word_idx["end"][: dataset.N], :]
        evals = torch.exp(vals)
        val_sum = torch.sum(evals, dim=1)
        assert val_sum.shape == (dataset.N,), val_sum.shape

        for key in ioi_dataset.word_idx.keys():
            end_to_s2 = att[
                torch.arange(dataset.N),
                ioi_dataset.word_idx["end"][: dataset.N],
                ioi_dataset.word_idx[key][: dataset.N],
            ]
            cur_ys.append(end_to_s2.mean().item())
            cur_stds.append(end_to_s2.std().item())
            average_attention[heads_raw][key] = end_to_s2.mean().item()
        fig.add_trace(
            go.Bar(
                x=list(ioi_dataset.word_idx.keys()),
                y=cur_ys,
                error_y=dict(type="data", array=cur_stds),
                name=str(heads_raw),
            )
        )
        fig.update_layout(
            title_text=f'Attention of NMs from END to various positions on {["ioi_dataset", "abc_dataset"][idx]}'
        )
    fig.show()

100%|██████████| 1/1 [00:08<00:00,  8.48s/it]


100%|██████████| 1/1 [00:08<00:00,  8.46s/it]


# Visualize attention patterns

In [ ]:
model.reset_hooks()
show_attention_patterns(model, [(9, 9), (9, 6), (10, 0)], ioi_dataset[:1])

/usr/local/lib/python3.12/dist-packages/easy_transformer/ioi_dataset.py:769: UserWarning:

Some groups have less than 5 prompts, they have lengths [1]



# Token and position signal results

In [ ]:
signal_specific_datasets = (
    {}
)  # keys are (token signal, positionnal signal) -1: inverted, 0: uncorrelated, 1: same as in ioi_dataset

# if ABB is the original pattern

signal_specific_datasets[(0, 1)] = ioi_dataset.gen_flipped_prompts(
    ("IO", "RAND")
).gen_flipped_prompts(
    ("S", "RAND")
)  # random name flip S1 and S2 are flipped to the same random name #DCC
signal_specific_datasets[(0, -1)] = signal_specific_datasets[
    (0, 1)
].gen_flipped_prompts(
    ("IO", "S1")
)  # CDC


signal_specific_datasets[(-1, -1)] = ioi_dataset.gen_flipped_prompts(
    ("S2", "IO")
)  # ABA
signal_specific_datasets[(-1, 1)] = signal_specific_datasets[
    (-1, -1)
].gen_flipped_prompts(
    ("IO", "S1")
)  # BAA


signal_specific_datasets[(1, -1)] = ioi_dataset.gen_flipped_prompts(("IO", "S1"))  # BAB
signal_specific_datasets[(1, 1)] = ioi_dataset  # ABB original dataset


def patch_end(z, source_act, hook):  # we patch at the "to" token
    z[torch.arange(ioi_dataset.N), ioi_dataset.word_idx["end"]] = source_act[
        torch.arange(ioi_dataset.N), ioi_dataset.word_idx["end"]
    ]
    return z


s_inhibition_heads = [(8, 6), (8, 10), (7, 3), (7, 9)]

logit_diff_per_signal = np.zeros((3, 2))

for k, source_dataset in signal_specific_datasets.items():

    config = PatchingConfig(
        source_dataset=source_dataset.toks.long(),
        target_dataset=ioi_dataset.toks.long(),
        target_module="attn_head",
        head_circuit="result",
        cache_act=True,
        verbose=False,
        patch_fn=patch_end,
        layers=(0, 9 - 1),
    )
    metric = ExperimentMetric(lambda x: x, ioi_dataset)  # dummy metric
    patching = EasyPatching(model, config, metric)

    model.reset_hooks()

    for l, h in s_inhibition_heads:
        hk_name, hk = patching.get_hook(
            l, h
        )  # we use the EasyPatching as a hook generator without running the experiment
        model.add_hook(hk_name, hk)

    tok_s, pos_s = k
    logit_diff_per_signal[tok_s + 1, (pos_s + 1) // 2] = logit_diff(model, ioi_dataset)


fig = px.imshow(logit_diff_per_signal)


fig.update_layout(
    yaxis=dict(
        tickmode="array",
        tickvals=[0, 1, 2],
        ticktext=[
            "Token signal inverted",
            "Token signal uncorrelated",
            "Token signal original",
        ],
    ),
    xaxis=dict(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Position signal inverted", "Position signal original"],
    ),
)
fig.show()

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
print("Memory cache cleared and garbage collected.")

Memory cache cleared and garbage collected.


# Backup NM results
 After ablating several attention heads, we actually an increase in logit difference

In [ ]:
model.reset_hooks()
default_logit_diff = logit_diff(model, ioi_dataset)
print(f"Recall that the initial logit diff is {default_logit_diff}")

top_name_movers = [(9, 9), (9, 6), (10, 0)]
exclude_heads = [(layer, head_idx) for layer in range(12) for head_idx in range(12)]
for head in top_name_movers:
    exclude_heads.remove(head)

the_extra_hooks = do_circuit_extraction(
    model=model,
    heads_to_remove=get_heads_circuit(
        ioi_dataset=ioi_dataset,
        circuit={"name mover": top_name_movers},
    ),
    mlps_to_remove={},
    ioi_dataset=ioi_dataset,
    mean_dataset=abc_dataset,
    return_hooks=True,
    excluded=exclude_heads,
)
model.reset_hooks()
for hook in the_extra_hooks:
    model.add_hook(*hook)
hooked_logit_diff = logit_diff(model, ioi_dataset)
print(
    f"After knocking out the three most important MLPs, logit diff is {hooked_logit_diff}"
)
model.reset_hooks()

both_results = []
pos = "end"

for idx, extra_hooks in enumerate([[], the_extra_hooks]):
    results = torch.zeros(size=(12, 12))
    mlp_results = torch.zeros(size=(12, 1))

    model.reset_hooks()
    for hook in extra_hooks:
        model.add_hook(*hook)
    hooked_logit_diff = logit_diff(model, ioi_dataset)
    model.reset_hooks()

    for source_layer in tqdm(range(12)):
        for source_head_idx in list(range(12)):
            model.reset_hooks()
            receiver_hooks = []
            receiver_hooks.append(("blocks.11.hook_resid_post", None))
            model = path_patching(
                model=model,
                D_new=abc_dataset,
                D_orig=ioi_dataset,
                sender_heads=[(source_layer, source_head_idx)],
                receiver_hooks=receiver_hooks,
                positions=[pos],
                return_hooks=False,
                extra_hooks=extra_hooks,
            )
            cur_logit_diff = logit_diff(model, ioi_dataset)

            if source_head_idx is None:
                mlp_results[source_layer] = cur_logit_diff - hooked_logit_diff
            else:
                results[source_layer][source_head_idx] = (
                    cur_logit_diff - hooked_logit_diff
                )

            if source_layer == 11 and source_head_idx == 11:
                fname = f"svgs/patch_and_freeze_{pos}_{ctime()}_{ri(2134, 123759)}"
                fig = show_pp(
                    results,
                    title=f"Direct effect of removing heads on logit diff"
                    + ("" if idx == 0 else " (with top 3 name movers knocked out)"),
                    return_fig=True,
                    show_fig=False,
                )

                both_results.append(results.clone())
                fig.show()

Recall that the initial logit diff is 3.626664638519287
After knocking out the three most important MLPs, logit diff is 3.408663749694824


  0%|          | 0/12 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1008.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 520.88 MiB is free. Process 4335 has 39.04 GiB memory in use. Of the allocated memory 38.32 GiB is allocated by PyTorch, and 234.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

 Plot the two sets of results

In [ ]:
from easy_transformer.ioi_utils import CLASS_COLORS
cc = deepcopy(CLASS_COLORS)
circuit = deepcopy(CIRCUIT)


def what_class(layer, head, circuit):
    for circuit_class in circuit:
        if (layer, head) in circuit[circuit_class]:
            return circuit_class
    return "duplicate token"


# plot the most important heads

for idx, results in enumerate(both_results):
    k = 15
    top_heads = max_2d(torch.abs(results), k=k)[  # backup results or initial results
        0
    ]  # initial results is the patch with no KOs; direct effect on logits

    exclude_heads = []
    exclude_heads = [
        (layer_idx, head)
        for layer_idx in range(12)
        for head in range(12)
        if what_class(layer_idx, head, circuit=circuit)
        not in ["name mover", "negative", "s2 inhibition"]
    ]

    fig = go.Figure()
    heights = [
        results[layer][head]
        for layer, head in top_heads
        if (layer, head) not in exclude_heads
    ]
    colors = [
        cc[what_class(layer, head_idx, circuit=circuit)]
        for layer, head_idx in top_heads
        if (layer, head_idx) not in exclude_heads
    ]

    # plot a bar chart
    fig.add_trace(
        go.Bar(
            x=[str(x) for x in top_heads if x not in exclude_heads],
            y=heights,
            orientation="v",
            marker_color=colors,
        )
    )

    # set y axis range to [-1, 1]
    fig.update_yaxes(range=[-3, 3])

    # update y axis
    fig.update_yaxes(title_text="Change in logit diffenrence after direct patching")

    # update title
    fig.update_layout(
        title="Most important heads by direct effect on logits"
        + ("" if idx == 0 else " (with top 3 name movers knocked out)")
    )
    fig.show()

# Validation outside of IOI
 Are the tasks of looking at previous tokens, inducting, and duplicating tokens performed on the general OWT distribution, rather than just p_IOI? Investigation of identified heads on random tokens

In [ ]:
seq_len = 100
rand_tokens = torch.randint(1000, 10000, (4, seq_len))
rand_tokens_repeat = einops.repeat(rand_tokens, "batch pos -> batch (2 pos)")


def calc_score(attn_pattern, hook, offset, arr):
    # Pattern has shape [batch, index, query_pos, key_pos]
    stripe = attn_pattern.diagonal(offset, dim1=-2, dim2=-1)
    scores = einops.reduce(stripe, "batch index pos -> index", "mean")
    # Store the scores in a common array
    arr[hook.layer()] = scores.detach().cpu().numpy()
    # return arr
    return attn_pattern


def filter_attn_hooks(hook_name):
    split_name = hook_name.split(".")
    return split_name[-1] == "hook_attn"


for mode, offset in [
    ("induction", 1 - seq_len),
    ("duplicate", -seq_len),
    ("previous", -1),
]:
    arr = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
    old_arr = deepcopy(arr)
    logits = model.run_with_hooks(
        rand_tokens_repeat,
        fwd_hooks=[(filter_attn_hooks, partial(calc_score, offset=offset, arr=arr))],
    )
    # print(torch.allclose(arr, old_arr))
    fig = px.imshow(
        arr,
        labels={"y": "Layer", "x": "Head"},
        color_continuous_scale="Blues",
    )
    fig.update_layout(title=f"Attention pattern for {mode} mode")
    fig.show()

In [ ]:
import torch

def proportion_s_logit_greater(model, dataset):
    # logits: [N, seq_len, vocab_size]
    logits = model(dataset.toks)

    count = 0
    for i in range(dataset.N):
        end_pos = dataset.word_idx["end"][i]

        # token IDs for IO and S at their positions
        io_pos = dataset.word_idx["IO"][i]
        s_pos  = dataset.word_idx["S"][i]
        io_tok = dataset.toks[i, io_pos]
        s_tok  = dataset.toks[i, s_pos]

        # compare logits at the END position
        if logits[i, end_pos, s_tok] > logits[i, end_pos, io_tok]:
            count += 1

    return count / dataset.N

print("Prop[S logit > IO] =", proportion_s_logit_greater(model, ioi_dataset))
